In [0]:
from pyspark.sql.functions import *

In [0]:
erp_bronze=spark.read.format("delta")\
    .load("s3://retail-lakehouse-ashu/bronze/erp/")

In [0]:
erp_bronze.groupBy(col("product_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("category.category_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("supplier.supplier_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("category.category_id").alias("category_id")) \
    .agg(
        countDistinct("category.category_name").alias("name_count")
    ) \
    .filter(col("name_count") > 1) \
    .show()

In [0]:
erp_bronze.groupBy(
    col("supplier.supplier_id").alias("supplier_id")
).agg(
    countDistinct("supplier.supplier_name").alias("name_count"),
    countDistinct("supplier.supplier_city").alias("city_count"),
    countDistinct("supplier.supplier_rating").alias("rating_count")
).filter(
    (col("name_count") > 1) |
    (col("city_count") > 1) |
    (col("rating_count") > 1)
).show(truncate=False)

In [0]:
erp_bronze.printSchema()

In [0]:
dim_product=erp_bronze.select(col("product_id"),col("category.category_id"),col("supplier.supplier_id"),col("product_name"),col("status"),col("pricing.cost_price"),col("pricing.selling_price"),col("created_date"),col("updated_date"),col("ingestion_timestamp"),col("batch_id"))
dim_product.display()

In [0]:
dim_category=erp_bronze.select(col("category.category_id"),col("category.category_name"),col("batch_id"))
dim_category.dropDuplicates(["category_id"]).display()

In [0]:
dim_supplier_silver=erp_bronze.select(col("supplier.supplier_id"),col("supplier.supplier_name"),col("supplier.supplier_city"),col("supplier.supplier_rating"),col("batch_id"))
dim_supplier_silver.dropDuplicates(["supplier_id"]).display()